In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from collections import Counter

In [3]:
class ID3:
    def __init__(self):
        self.tree = None

    def entropy(self, y):
        probs = y.value_counts(normalize=True)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def information_gain(self, data, feature, target):
        total_entropy = self.entropy(data[target])
        values = data[feature].unique()

        weighted_entropy = 0
        for val in values:
            subset = data[data[feature] == val]
            weighted_entropy += (len(subset) / len(data)) * self.entropy(subset[target])

        return total_entropy - weighted_entropy

    def build_tree(self, data, features, target):
        y = data[target]

        # Если все объекты одного класса
        if len(y.unique()) <= 1:
            return y.iloc[0]

        # Если признаков больше нет — возвращаем самый частый класс
        if not features:
            return y.value_counts().idxmax()

        # Выбираем лучший признак по Information Gain
        gains = {f: self.information_gain(data, f, target) for f in features}
        best_feature = max(gains, key=gains.get)

        tree = {best_feature: {}}
        remaining_features = [f for f in features if f != best_feature]

        # Рекурсивно строим ветви для каждого значения признака
        for val in data[best_feature].unique():
            subset = data[data[best_feature] == val]
            tree[best_feature][val] = self.build_tree(subset, remaining_features, target)

        return tree

    def fit(self, X, y):
        data = X.copy()
        target_name = 'target'
        data[target_name] = y
        self.tree = self.build_tree(data, list(X.columns), target_name)

    def predict_one(self, sample, tree):
        if not isinstance(tree, dict):
            return tree
        feature = list(tree.keys())[0]
        value = sample[feature]
        if value not in tree[feature]:
            return None # Значение не встречалось при обучении
        return self.predict_one(sample, tree[feature][value])

    def predict(self, X):
        return X.apply(lambda row: self.predict_one(row, self.tree), axis=1)


In [4]:
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)

In [5]:
# Дискретизация: превращаем числа в 3 категории (bins)
for col in df.columns:
    df[col] = pd.cut(df[col], bins=3, labels=['low', 'medium', 'high'])

y = pd.Series(iris.target)

In [6]:
# Обучение
model = ID3()
model.fit(df, y)

In [7]:
import pprint

# Результат
print("Построенное дерево:")
pprint.pprint(model.tree)

Построенное дерево:
{'petal width (cm)': {'high': {'petal length (cm)': {'high': np.int64(2),
                                                     'medium': {'sepal width (cm)': {'low': np.int64(2),
                                                                                     'medium': {'sepal length (cm)': {'medium': np.int64(2)}}}}}},
                      'low': np.int64(0),
                      'medium': {'petal length (cm)': {'high': {'sepal length (cm)': {'high': np.int64(2),
                                                                                      'medium': {'sepal width (cm)': {'low': np.int64(2),
                                                                                                                      'medium': np.int64(1)}}}},
                                                       'medium': {'sepal length (cm)': {'high': np.int64(1),
                                                                                        'low': {'sepal width (cm)

In [8]:
# Проверка на одном примере
test_sample = df.iloc[0]
prediction = model.predict_one(test_sample, model.tree)
print(f"\nПредсказание для первого примера: {prediction} (Реальный класс: {y[0]})")


Предсказание для первого примера: 0 (Реальный класс: 0)


In [9]:
from sklearn.model_selection import train_test_split

# Подготовка данных (используем дискретизированный ирис) и разбиение
X = df
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Инициализация и обучение модели
model_id3 = ID3()
model_id3.fit(X_train, y_train)


In [11]:
# Получение предсказаний для тестовой выборки
predictions = model_id3.predict(X_test)

In [12]:
# Функция для расчета точности (Accuracy)
def calculate_accuracy(y_true, y_pred):
    # Убираем None, если модель встретила неизвестное значение признака
    valid_indices = y_pred.notnull()
    correct = (y_true[valid_indices] == y_pred[valid_indices]).sum()
    return correct / len(y_true)

accuracy = calculate_accuracy(y_test, predictions)

print(f"Точность (Accuracy) на тестовой выборке: {accuracy:.2%}")

# Сравнение первых 10 результатов
results_comparison = pd.DataFrame({'Реальность': y_test, 'Предсказание': predictions})
print("\nСравнение первых 10 ответов:")
print(results_comparison.head(10))

Точность (Accuracy) на тестовой выборке: 100.00%

Сравнение первых 10 ответов:
     Реальность  Предсказание
73            1             1
18            0             0
118           2             2
78            1             1
76            1             1
31            0             0
64            1             1
141           2             2
68            1             1
82            1             1


In [13]:
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y)

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _information_gain(self, y, X_column, split_thresh):
        parent_entropy = self._entropy(y)
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_entropy - child_entropy

    def _best_criteria(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Остановка, если достигнута глубина или чистота узла
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return {'leaf': True, 'value': leaf_value}

        feat_idxs = np.random.choice(n_features, n_features, replace=False)
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return {'leaf': False, 'feature': best_feat, 'threshold': best_thresh, 'left': left, 'right': right}

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node['leaf']:
            return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._traverse_tree(x, node['left'])
        return self._traverse_tree(x, node['right'])

In [14]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            # Бутстрап (выборка с возвращением)
            indices = np.random.choice(len(X), len(X), replace=True)
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Мажоритарное голосование
        return np.array([Counter(tree_preds[:, i]).most_common(1)[0][0] for i in range(X.shape[0])])

In [15]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
clf = RandomForest(n_trees=5)
clf.fit(X_train, y_train)

In [17]:
predictions = clf.predict(X_test)
accuracy = np.sum(predictions == y_test) / len(y_test)

In [18]:
print(f"Точность (Accuracy): {accuracy * 100:.2f}%")

Точность (Accuracy): 100.00%


# Лабораторная работа. Rain in Australia

https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package

Вы когда-нибудь задумывались, стоит ли завтра брать с собой зонт? С помощью этого набора данных вы можете предсказать дождь на следующий день, обучив модели классификации на целевой переменной RainTomorrow.

Содержание
Этот набор данных включает в себя примерно 10 лет ежедневных наблюдений за погодой из многочисленных мест по всей Австралии.

Переменная RainTomorrow является целевой переменной для прогнозирования. Она отвечает на важнейший вопрос: будет ли дождь на следующий день? (Да или Нет).

Вам необходимо выбрать наилучший алгоритм для решения задачи бинарной классификации на данном датасете среди: C4.5, CART, Случайного леса.

5 баллов будут даны тем студентам, чье значение точности модели будет в пределах 10% от наивысшего по группе.

In [ ]:
import pandas as pd

In [ ]:
# Загрузка датасета
au_rain = pd.read_csv('/content/weatherAUS.csv')

In [ ]:
print(au_rain.head())

In [ ]:
# Предобработка данных и разделение на обучающую и тестовую выборки

In [ ]:
# Наиболее близкая модель к C4.5
model1 = DecisionTreeClassifier(criterion='entropy')

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# CART для классификации (использует Gini по умолчанию)
model2 = DecisionTreeClassifier()

In [ ]:
# Случайный лес
from sklearn.ensemble import RandomForestClassifier
model3 = RandomForestClassifier(n_estimators=100) # 100 деревьев